In [28]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time


In [29]:
# Setup Chrome options
chrome_options = Options()
# chrome_options.add_argument("--headless")  # Uncomment for headless mode

# Initialize the driver
driver = webdriver.Chrome(options=chrome_options)

# Navigate to the NCSL AI legislation database
url = "https://www.ncsl.org/financial-services/artificial-intelligence-legislation-database"
driver.get(url)

# Wait for page to load
wait = WebDriverWait(driver, 15)

# Wait for the checkboxes to be present
time.sleep(3)  # Allow dynamic content to load

# Locate "All Topics" checkbox
all_topics_checkbox = wait.until(
    EC.presence_of_element_located((By.XPATH, "//label[contains(text(), 'All Topics')]/preceding-sibling::input | //label[contains(text(), 'All Topics')]//input | //input[@type='checkbox'][following-sibling::*[contains(text(), 'All Topics')] or preceding-sibling::*[contains(text(), 'All Topics')]]"))
)
print(f"All Topics checkbox found: {all_topics_checkbox.get_attribute('outerHTML')}")

# Locate "All States" checkbox
all_states_checkbox = wait.until(
    EC.presence_of_element_located((By.XPATH, "//label[contains(text(), 'All States')]/preceding-sibling::input | //label[contains(text(), 'All States')]//input | //input[@type='checkbox'][following-sibling::*[contains(text(), 'All States')] or preceding-sibling::*[contains(text(), 'All States')]]"))
)
print(f"All States checkbox found: {all_states_checkbox.get_attribute('outerHTML')}")

print("\nBoth checkboxes located successfully.")


All Topics checkbox found: <input id="dnn_ctr34167_StateNetDB_ckBxAllTopics" type="checkbox" name="dnn$ctr34167$StateNetDB$ckBxAllTopics">
All States checkbox found: <input id="dnn_ctr34167_StateNetDB_ckBxAllStates" type="checkbox" name="dnn$ctr34167$StateNetDB$ckBxAllStates">

Both checkboxes located successfully.


In [30]:
# Step 1: Update YEAR dropdown to 2025
from selenium.webdriver.support.ui import Select

# First find all select elements and print their IDs to identify the correct one
all_selects = driver.find_elements(By.TAG_NAME, "select")
print("Found select elements:")
for sel in all_selects:
    print(f"  ID: {sel.get_attribute('id')}, Name: {sel.get_attribute('name')}")

# Find YEAR dropdown (ID: dnn_ctr34167_StateNetDB_ddlYear)
year_dropdown = driver.find_element(By.ID, "dnn_ctr34167_StateNetDB_ddlYear")
year_select = Select(year_dropdown)
year_select.select_by_visible_text("2025")
print("Step 1: Year updated to 2025")

# Step 2: Check the "All Topics" and "All States" checkboxes
# Scroll to element and use JavaScript click to avoid interception issues
all_topics_checkbox = driver.find_element(By.ID, "dnn_ctr34167_StateNetDB_ckBxAllTopics")
driver.execute_script("arguments[0].scrollIntoView(true);", all_topics_checkbox)
time.sleep(0.5)
if not all_topics_checkbox.is_selected():
    driver.execute_script("arguments[0].click();", all_topics_checkbox)
print("Step 2a: All Topics checkbox checked")

all_states_checkbox = driver.find_element(By.ID, "dnn_ctr34167_StateNetDB_ckBxAllStates")
driver.execute_script("arguments[0].scrollIntoView(true);", all_states_checkbox)
time.sleep(0.5)
if not all_states_checkbox.is_selected():
    driver.execute_script("arguments[0].click();", all_states_checkbox)
print("Step 2b: All States checkbox checked")

# Step 3: Click the Search button
# First find buttons to identify correct ID
all_buttons = driver.find_elements(By.XPATH, "//input[@type='submit'] | //input[@type='button'] | //button")
print("Found buttons:")
for btn in all_buttons:
    print(f"  ID: {btn.get_attribute('id')}, Value: {btn.get_attribute('value')}, Text: {btn.text}")

# Use the button with value='Search' following same pattern as other elements
search_button = driver.find_element(By.ID, "dnn_ctr34167_StateNetDB_btnSearch")
driver.execute_script("arguments[0].scrollIntoView(true);", search_button)
time.sleep(0.5)
driver.execute_script("arguments[0].click();", search_button)
print("Step 3: Search button clicked")

# Step 4: Wait for 3 seconds
time.sleep(3)
print("Step 4: Waited 3 seconds")


Found select elements:
  ID: dnn_ctr34167_StateNetDB_ddlStatus, Name: dnn$ctr34167$StateNetDB$ddlStatus
  ID: dnn_ctr34167_StateNetDB_ddlYear, Name: dnn$ctr34167$StateNetDB$ddlYear
  ID: dnn_ctr12532_View_Dropdown_12532_5, Name: dnn$ctr12532$View$Dropdown_12532_5
  ID: dnn_ctr12532_View_Dropdown_12532_6, Name: dnn$ctr12532$View$Dropdown_12532_6
Step 1: Year updated to 2025
Step 2a: All Topics checkbox checked
Step 2b: All States checkbox checked
Found buttons:
  ID: , Value: , Text: Preferences
  ID: , Value: , Text: Decline
  ID: , Value: , Text: Accept
  ID: , Value: , Text: 
  ID: search-toggle, Value: , Text: 
  ID: findresults, Value: , Text: 
  ID: btnClearAllTopics, Value: , Text: Clear
  ID: btnClearAllStates, Value: , Text: Clear
  ID: dnn_ctr34167_StateNetDB_btnSearch, Value: Search, Text: 
  ID: dnn_ctr34167_StateNetDB_btnReset, Value: Reset All, Text: 
  ID: , Value: , Text: Topic Descriptions
  ID: , Value: , Text: 
  ID: dnn_ctr12532_View_Submitbutton_12532_9, Value: Subm

In [ ]:
# Inspect the results structure - first get the full page source to find actual elements
time.sleep(5)  # Wait longer for results to load

# Get page source and look for results-related elements
page_source = driver.page_source

# Search for key patterns in the HTML
import re

# Find elements containing "RESULTS" or "Total Bills"
print("=== Searching page source for results patterns ===\n")

# Look for "Total Bills" pattern
if "Total Bills" in page_source:
    # Find the surrounding context
    idx = page_source.find("Total Bills")
    print(f"Found 'Total Bills' at index {idx}")
    print(f"Context: ...{page_source[max(0,idx-200):idx+200]}...")

# Find all div IDs containing 'Result' or 'pnl'
div_pattern = r'id="([^"]*(?:Result|pnl|results)[^"]*)"'
matches = re.findall(div_pattern, page_source, re.IGNORECASE)
print(f"\nFound IDs containing 'Result/pnl/results': {matches[:20]}")

# Find elements with "ALABAMA" (first state in results)
if "ALABAMA" in page_source:
    idx = page_source.find("ALABAMA")
    print(f"\nFound 'ALABAMA' at index {idx}")
    print(f"Context (500 chars before): ...{page_source[max(0,idx-500):idx+100]}...")

# Print all panel/container IDs
panel_pattern = r'id="(dnn_ctr34167[^"]*)"'
panel_matches = re.findall(panel_pattern, page_source)
print(f"\nAll dnn_ctr34167 IDs found: {panel_matches}")

=== Searching page source for results patterns ===

Found 'Total Bills' at index 2075737
Context: ...TS
    <div id="dnn_ctr34167_StateNetDB_resultsCount" style="float: right; color: White">
        <span id="dnn_ctr34167_StateNetDB_lblResultCount">Total States: 53&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Total Bills: 1262</span></div>
</div>
<a id="dnn_ctr34167_StateNetDB_HyperLink1" href="//www.ncsl.org"></a>
<span id="dnn_ctr34167_StateNetDB_lblNoResults"></span>
<div id="dnn_ctr34167_StateNetDB_lin...

Found IDs containing 'Result/pnl/results': ['findresults', 'dnn_ctr34167_StateNetDB_resultsHeader', 'dnn_ctr34167_StateNetDB_resultsCount', 'dnn_ctr34167_StateNetDB_lblResultCount', 'dnn_ctr34167_StateNetDB_lblNoResults', 'dnn_ctr34167_StateNetDB_repResults_divHistoryList_0', 'dnn_ctr34167_StateNetDB_repResults_divHistoryList_1', 'dnn_ctr34167_StateNetDB_repResults_divHistoryList_2', 'dnn_ctr34167_StateNetDB_repResults_divHistoryList_3', 'dnn_ctr34167_StateNetDB_repResults_divHistoryList_4

In [33]:
# Get the HTML content of the first bill element
first_bill = driver.find_element(By.ID, "dnn_ctr34167_StateNetDB_repResults_divHistoryList_0")
print(first_bill.get_attribute('outerHTML'))


<div id="dnn_ctr34167_StateNetDB_repResults_divHistoryList_0" class="historyList" style="display: none">
                02/05/2025 - INTRODUCED.<br>02/05/2025 - To HOUSE Committee on WAYS AND MEANS EDUCATION.
            </div>


In [34]:
# Explore parent and sibling elements to locate all bill fields
first_history = driver.find_element(By.ID, "dnn_ctr34167_StateNetDB_repResults_divHistoryList_0")

# Get parent element
parent = first_history.find_element(By.XPATH, "..")
print("=== Parent element ===")
print(f"Tag: {parent.tag_name}")
print(f"Class: {parent.get_attribute('class')}")
print(f"ID: {parent.get_attribute('id')}")
print(f"\nParent outerHTML (first 3000 chars):\n{parent.get_attribute('outerHTML')[:3000]}")

# Get grandparent to see more context
grandparent = parent.find_element(By.XPATH, "..")
print("\n\n=== Grandparent element ===")
print(f"Tag: {grandparent.tag_name}")
print(f"Class: {grandparent.get_attribute('class')}")
print(f"ID: {grandparent.get_attribute('id')}")
print(f"\nGrandparent outerHTML (first 5000 chars):\n{grandparent.get_attribute('outerHTML')[:5000]}")


=== Parent element ===
Tag: div
Class: 
ID: dnn_ctr34167_StateNetDB_linkList

Parent outerHTML (first 3000 chars):
<div id="dnn_ctr34167_StateNetDB_linkList" style="width: 99%; text-align: left; margin-top: 10px">
    
            <div class="h2Headers1">Alabama<div style="float: right; padding-right: 5px;"></div></div>
            <div style="clear: both">
            </div>
            <div>
                <a href="http://custom.statenet.com/public/resources.cgi?id=ID:bill:AL2025000H169&amp;ciq=urn:user:PA196471263&amp;client_md=5883624f12ed272ea6ef90117e0c6cca&amp;mode=current_text" target="_blank">AL H    169</a><br>
                2025<br>
            </div>
            <div style="font-weight: bold;">
                Public Education<br>
            </div>
            <b>Status:</b>
            Failed - Adjourned - House Ways and Means Education Committee
            <br>
            <b>Date of Last Action:</b>
            2/5/2025
            <br>
            <b>Author:</b>
  

In [35]:
import json

# Get the linkList container
link_list = driver.find_element(By.ID, "dnn_ctr34167_StateNetDB_linkList")

# Extract first bill data by parsing the HTML content
first_bill_data = {}

# State header
state_header = link_list.find_element(By.CLASS_NAME, "h2Headers1")
first_bill_data["state"] = state_header.text.strip()

# Bill ID and link
bill_link = link_list.find_element(By.XPATH, ".//a[contains(@href, 'statenet.com')]")
first_bill_data["bill_id"] = bill_link.text.strip()
first_bill_data["bill_url"] = bill_link.get_attribute("href")

# Title (bold div)
title_div = link_list.find_element(By.XPATH, ".//div[@style='font-weight: bold;']")
first_bill_data["title"] = title_div.text.strip()

# Extract text fields by finding <b> tags and getting following text
def get_field_value(container, field_name):
    try:
        b_tag = container.find_element(By.XPATH, f".//b[contains(text(), '{field_name}')]")
        # Get the parent's text and extract value after the label
        parent = b_tag.find_element(By.XPATH, "..")
        full_text = parent.get_attribute("innerHTML")
        # Split by the </b> tag and get the value part
        parts = full_text.split("</b>")
        if len(parts) > 1:
            value = parts[1].split("<br>")[0].strip()
            return value
    except:
        return None
    return None

first_bill_data["status"] = get_field_value(link_list, "Status:")
first_bill_data["date_of_last_action"] = get_field_value(link_list, "Date of Last Action:")
first_bill_data["author"] = get_field_value(link_list, "Author:")
first_bill_data["topics"] = get_field_value(link_list, "Topics:")
first_bill_data["summary"] = get_field_value(link_list, "Summary:")

# History
history_div = link_list.find_element(By.ID, "dnn_ctr34167_StateNetDB_repResults_divHistoryList_0")
first_bill_data["history"] = history_div.get_attribute("innerHTML").replace("<br>", "\n").strip()

print(json.dumps(first_bill_data, indent=2, ensure_ascii=False))


{
  "state": "ALABAMA",
  "bill_id": "AL H 169",
  "bill_url": "http://custom.statenet.com/public/resources.cgi?id=ID:bill:AL2025000H169&ciq=urn:user:PA196471263&client_md=5883624f12ed272ea6ef90117e0c6cca&mode=current_text",
  "title": "Public Education",
  "status": "Failed - Adjourned - House Ways and Means Education Committee",
  "date_of_last_action": "Failed - Adjourned - House Ways and Means Education Committee",
  "author": "Failed - Adjourned - House Ways and Means Education Committee",
  "topics": "Failed - Adjourned - House Ways and Means Education Committee",
  "summary": "Failed - Adjourned - House Ways and Means Education Committee",
  "history": "02/05/2025 - INTRODUCED.\n02/05/2025 - To HOUSE Committee on WAYS AND MEANS EDUCATION."
}


In [42]:
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

# Step 1 & 3: Scroll to load all content and get full HTML
link_list = driver.find_element(By.ID, "dnn_ctr34167_StateNetDB_linkList")
driver.execute_script("arguments[0].scrollIntoView(false);", link_list)
time.sleep(1)

# Scroll through the page to ensure all content is loaded
for i in tqdm(range(10), desc="Scrolling"):
    driver.execute_script("window.scrollBy(0, 2000);")
    time.sleep(0.3)

# Get the full HTML content
html_content = link_list.get_attribute("innerHTML")
soup = BeautifulSoup(html_content, "html.parser")

# Step 2: Extract all bills
all_bills = []
current_state = None

# Find all elements in order
elements = soup.find_all(["div", "b", "a", "hr"])

i = 0
bill_data = {}
with tqdm(total=len(elements), desc="Parsing bills") as pbar:
    while i < len(elements):
        elem = elements[i]
        
        # State header
        if elem.name == "div" and elem.get("class") and "h2Headers1" in elem.get("class"):
            current_state = elem.get_text(strip=True)
            i += 1
            pbar.update(1)
            continue
        
        # Bill link (starts a new bill)
        if elem.name == "a" and elem.get("href") and "statenet.com" in elem.get("href", ""):
            # Save previous bill if exists
            if bill_data:
                all_bills.append(bill_data)
            
            bill_data = {
                "state": current_state,
                "bill_id": elem.get_text(strip=True),
                "bill_url": elem.get("href"),
                "title": None,
                "status": None,
                "date_of_last_action": None,
                "author": None,
                "topics": None,
                "summary": None,
                "history": None
            }
            i += 1
            pbar.update(1)
            continue
        
        # Title (bold div)
        if elem.name == "div" and elem.get("style") and "font-weight: bold" in elem.get("style", ""):
            if bill_data:
                bill_data["title"] = elem.get_text(strip=True)
            i += 1
            pbar.update(1)
            continue
        
        # Field labels
        if elem.name == "b":
            label = elem.get_text(strip=True)
            # Get next sibling text
            next_text = elem.next_sibling
            if next_text:
                value = str(next_text).strip()
                if label == "Status:":
                    bill_data["status"] = value
                elif label == "Date of Last Action:":
                    bill_data["date_of_last_action"] = value
                elif label == "Author:":
                    bill_data["author"] = value
                elif label.startswith("Topics"):
                    bill_data["topics"] = value
                elif label.startswith("Summary"):
                    bill_data["summary"] = value
            i += 1
            pbar.update(1)
            continue
        
        # History div
        if elem.name == "div" and elem.get("id") and "divHistoryList" in elem.get("id", ""):
            if bill_data:
                bill_data["history"] = elem.get_text(separator="\n", strip=True)
            i += 1
            pbar.update(1)
            continue
        
        i += 1
        pbar.update(1)

# Save last bill
if bill_data and bill_data.get("bill_id"):
    all_bills.append(bill_data)

# Step 4: Convert to DataFrame
df = pd.DataFrame(all_bills)

# 1. Shape
print(f"Shape: {df.shape}")

# 2. Head/Tail
print("\nHead:")
print(df.head(3).to_string())
print("\nTail:")
print(df.tail(3).to_string())

# 3. Columns
print(f"\nColumns: {list(df.columns)}")

# 4. NaN counts
print(f"\nNaN counts:\n{df.isna().sum()}")


Parsing bills: 100%|██████████| 18305/18305 [00:00<00:00, 427615.96it/s]

Shape: (1262, 10)

Head:
     state      bill_id                                                                                                                                                        bill_url             title                                                             status date_of_last_action        author                         topics                                                                                                                                                                                       summary                                                                                     history
0  Alabama  AL H    169  http://custom.statenet.com/public/resources.cgi?id=ID:bill:AL2025000H169&ciq=urn:user:PA196471263&client_md=5883624f12ed272ea6ef90117e0c6cca&mode=current_text  Public Education      Failed - Adjourned - House Ways and Means Education Committee            2/5/2025   Garrett (R)  Appropriations, Education Use  Relates to make appropri

In [45]:
# Save to CSV with UTF-8 encoding
import os
os.makedirs("data/ncsl", exist_ok=True)
output_path = "data/ncsl/us_ai_legislation_ncsl_meta.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")



In [ ]:
import pandas as pd
import requests
import json
import os
from tqdm.auto import tqdm

# Step 1: Read CSV
df = pd.read_csv("data/ncsl/us_ai_legislation_ncsl_meta.csv")
print(f"Loaded {len(df)} rows")

# Output path
os.makedirs("data/ncsl", exist_ok=True)
output_path = "data/ncsl/ncsl_bills.jsonl"

# Step 2-5: Iterate, fetch HTML, save to JSONL
with open(output_path, "w", encoding="utf-8") as f:
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Fetching bills"):
        bill_id = row["bill_id"]
        bill_url = row["bill_url"]
        
        try:
            # Step 4: Fetch HTML with requests
            resp = requests.get(bill_url, timeout=30)
            resp.raise_for_status()
            html = resp.text
        except Exception as e:
            html = f"ERROR: {str(e)}"
        
        # Step 5: Write to JSONL
        record = {"bill_id": bill_id, "html": html}
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


Loaded 1262 rows


Fetching bills:   4%|▍         | 54/1262 [00:41<20:52,  1.04s/it]

In [50]:
# check JSONL
import json

JSONL_PATH = "data/ncsl/ncsl_bills.jsonl"
lines = open(JSONL_PATH, encoding="utf-8").readlines()
records = [json.loads(l) for l in lines]

print("rows:", len(records))
print("cols:", list(records[0].keys()) if records else None)
print("errors:", sum(1 for r in records if str(r.get("html","")).startswith("ERROR:")))

print("\nhead:")
for r in records[:3]:
    print(r["bill_id"], len(r.get("html","")))

print("\ntail:")
for r in records[-3:]:
    print(r["bill_id"], len(r.get("html","")))


rows: 1262
cols: ['bill_id', 'html']
errors: 0

head:
AL H    169 329010
AL H    208 8848
AL H    235 5664

tail:
WY SJR  8 6643
WY S    30 9528
WY D    149 9070


In [51]:
# dump first row
import json
with open("data/ncsl/ncsl_bills.jsonl", encoding="utf-8") as f:
    first = json.loads(f.readline())
print(json.dumps(first, indent=2, ensure_ascii=False)[:5000])

{
  "bill_id": "AL H    169",
  "html": "<html>\n<head>\n<title>Bill Resource</title>\n\n<!--   <link href=\"https://custom.statenet.com/network/Common/css/extregtext.css\" rel=\"stylesheet\" type=\"text/css\" />-->\n   <link href=\"https://custom.statenet.com/network/Common/css/xmltext-2.0.css\" rel=\"stylesheet\" type=\"text/css\" />\n   <link href=\"https://custom.statenet.com/network/Common/css/additional-text.css\" rel=\"stylesheet\" type=\"text/css\" />\n\n\t<style type=\"text/css\">\n<!--\n\ntd, body {\n     background-color: white;\n\tfont-family: Verdana, Arial, Helvetica, sans-serif;\n\tfont-size: 12px;\n}\n\n.resourceContainer {\n   border: 1px solid black;\n}\n-->\n</style>\n</head>\n<body>\n\n    <div style=\"width: 750px; margin: auto\">\n       <div style=\"font-size: .8em;margin-bottom: 10px\"><table width=\"100%\"><tr><td align=\"left\" style=\"font-size: .8em;\"><div>The following has special meaning:</div>\n<div><u class=\"amendmentInsertedText\">green underline deno

In [52]:
# inspect HTML structure
from bs4 import BeautifulSoup
import json

with open("data/ncsl/ncsl_bills.jsonl", encoding="utf-8") as f:
    rec = json.loads(f.readline())

soup = BeautifulSoup(rec["html"], "html.parser")

# print all tags with id or class
for tag in soup.find_all(True):
    attrs = []
    if tag.get("id"):
        attrs.append(f'id="{tag["id"]}"')
    if tag.get("class"):
        attrs.append(f'class="{" ".join(tag["class"])}"')
    if attrs:
        print(f"<{tag.name} {' '.join(attrs)}>")
    else:
        print(f"<{tag.name}>")

<html>
<head>
<title>
<link>
<link>
<style>
<body>
<div>
<div>
<table>
<tr>
<td>
<div>
<div>
<u class="amendmentInsertedText">
<div>
<strike class="amendmentDeletedText">
<td>
<a>
<img>
<table id="text-identifier">
<tr>
<td class="key">
<td>
<table>
<tr>
<td class="label">
<td>
<tr>
<td class="label">
<td>
<tr>
<td class="label">
<td>
<div class="documentBody">
<a>
<div class="head">
<p class="indent">
<p class="left">
<p class="left">
<p class="left">
<p class="left">
<p class="center">
<p class="left">
<a>
<div class="title">
<p class="center">
<p class="center">
<p class="center">
<p class="indent">
<a>
<div class="text">
<span>
<p class="left">
<p class="indent">
<p class="indent">
<p class="indent">
<p class="indent">
<p class="indent">
<table>
<tr>
<td>
<td>
<td>
<td>
<tr>
<td>
<td>
<td>
<td>
<tr>
<td>
<td>
<td>
<td>
<tr>
<td>
<td>
<td>
<td>
<tr>
<td>
<td>
<td>
<td>
<tr>
<td>
<td>
<td>
<td>
<tr>
<td>
<td>
<td>
<td>
<tr>
<td>
<td>
<td>
<td>
<tr>
<td>
<td>
<td>
<td>
<tr>
<td>
<td>
